# Avance 1 — Asistente para la identificación de servicios REST

Chatbot basado en reglas con Python 3.10+ y NLTK.

**Universidad Tecmilenio** · Desarrollo de aplicaciones con inteligencia artificial  
**Alumno:** Alejandro López Arteaga · **Matrícula:** 07302961

Funciona en VS Code (extensión Jupyter, abierto desde la carpeta del repositorio) y en Google Colab (clona el repositorio).

## 1. Entorno

In [ ]:
import sys, os
print(sys.version)
EN_COLAB = 'google.colab' in sys.modules
# En Colab se clona el repositorio una sola vez; al volver a ejecutar la celda sólo se entra a la carpeta
if EN_COLAB and not os.path.exists('chatbot_servicios.py'):
    if not os.path.exists('chatbot-tm'):
        !git clone -q https://github.com/lopartale/chatbot-tm.git
    os.chdir('chatbot-tm')
%pip install -q "nltk>=3.9" "PyYAML>=6.0" "pandas>=2.0"
import nltk
print('NLTK', nltk.__version__)

## 2. Base de conocimiento (inventario leído desde un archivo RAML)

In [ ]:
from inventario_raml import cargar_inventario_raml
inv = cargar_inventario_raml('data/plataforma_servicios_bancarios.raml')
print(inv['titulo'], inv['version'], '-', len(inv['servicios']), 'servicios')
for s in inv['servicios']:
    print(f"{s['id']}  {s['dominio']:<15} {s['metodo']:<6} {s['endpoint']}")

## 3. Arquitectura: recorrido de un mensaje
Cada mensaje pasa por tres componentes: preprocesamiento (normalización, tokenización, palabras vacías, sinónimos y stemming), clasificador de intenciones (`nltk.chat.util.Chat` y reglas regex ordenadas) y gestor de respuestas (búsqueda ponderada en el inventario, cobertura, brechas y contexto de sesión).

En cada turno el mensaje sigue este flujo: normalizar → ¿charla? → reglas de intención en orden → (si ninguna regla aplica) ¿coincide con el inventario? → manejador de la intención ↔ base de conocimiento → respuesta + contexto.

In [ ]:
from chatbot_servicios import AsistenteServiciosREST
from preprocesamiento import preprocesar

bot = AsistenteServiciosREST()
for mensaje in ['Hola', 'Necesito una API de login para que el usuario pueda ingresar',
                'movimientos de la cuenta del último mes', 'Cuéntame un chiste']:
    print('Mensaje:', mensaje)
    preprocesado = preprocesar(mensaje)                                       # Normalizar
    print('  Normalizado:', repr(preprocesado.texto), '| raíces:', preprocesado.raices)
    if bot.clasificador.charla(preprocesado):                                 # ¿Charla?
        print('  ¿Charla? sí')
    else:
        clasificacion = bot.clasificador.aplicar_reglas(preprocesado)         # Reglas de intención en orden
        if clasificacion:
            print('  Regla de intención:', clasificacion.intencion)
        else:
            coincide = bot.gestor.coincide_con_inventario(preprocesado)       # ¿Coincide con el inventario?
            print('  Ninguna regla -> ¿coincide con el inventario?', 'sí' if coincide else 'no')
    respuesta = bot.responder(mensaje)                                        # el turno completo
    print('  Intención final:', bot.historial[-1]['intencion'])
    print('  Respuesta:', respuesta.splitlines()[0])
    print('  Contexto (últimos servicios):', bot.ultimos_recomendados)

## 4. Pruebas funcionales y de generalización

In [ ]:
import pruebas_conversaciones as pc
res = pc.ejecutar()
pc.prueba_generalizacion()

In [ ]:
import pandas as pd
df = pd.DataFrame(res)
print(df.groupby('esperada')['correcto'].agg(['count', 'sum']))
print('Precisión global:', round(df['correcto'].mean() * 100, 1), '%')

## 5. Modo interactivo
Escribe `salir` para terminar.

In [ ]:
from chatbot_servicios import main
main()